In [5]:
import os
import sys
import json
import random
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any
from collections import defaultdict

# Наукові обчислення та обробка даних
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Computer Vision та обробка зображень
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pycocotools import mask as maskUtils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# PyTorch - основний фреймворк
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Sampler
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import nms, box_iou

# Аугментації
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Перевірка CUDA
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
print(f"Training device: {device}")

PyTorch version: 2.8.0+cu129
CUDA available: True
CUDA device: NVIDIA GeForce RTX 3060
CUDA version: 12.9
Training device: cuda


In [6]:
@dataclass
class Config:
    coco_root: str = '../coco2017'
    train_root: str = os.path.join(coco_root, 'train2017')
    val_root: str = os.path.join(coco_root, 'val2017')
    train_ann: str = os.path.join(coco_root, 'annotations', 'instances_train2017.json')
    val_ann: str = os.path.join(coco_root, 'annotations', 'instances_val2017.json')

    # Device
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seed: int = 42
    # Checkpoint
    checkpoint_dir: str = './checkpoints'
    save_every: int = 5  # epochs

cfg = Config()
print("\n" + "="*80)
print("КОНФІГУРАЦІЯ ПРОЕКТУ")
print("="*80)
print(f"Dataset: COCO 2017")
print(f"Device: {cfg.device}")
print(f"Checkpoint dir: {cfg.checkpoint_dir}")
print("="*80 + "\n")    


КОНФІГУРАЦІЯ ПРОЕКТУ
Dataset: COCO 2017
Device: cuda
Checkpoint dir: ./checkpoints



In [7]:
"""
Враховуючи аналіз з Частини 1 (lab1.ipynb):
- 80 класів COCO
- Середній розмір bbox: ~118x118 пікселів  
- Aspect ratio об'єктів: переважно 0.5-2.0
- Багато малих об'єктів (area < 32²)
"""

class AlexNetBackbone(nn.Module):
    """
    AlexNet feature extractor (оригінальні conv layers 1-5).
    
    Вхід: [B, 3, H, W] RGB зображення
    Вихід: [B, 256, H/16, W/16] feature map
    """
    def __init__(self, pretrained: bool = True):
        super().__init__()
        
        # Завантажуємо оригінальний AlexNet
        if pretrained:
            alexnet = torchvision.models.alexnet(weights=torchvision.models.AlexNet_Weights.DEFAULT)
        else:
            alexnet = torchvision.models.alexnet(weights=None)
        
        # Беремо тільки convolutional частину (features)
        self.features = alexnet.features
        
        # Додаткові conv шари для покращення feature map
        self.extra_conv = nn.Sequential(
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
        
    def forward(self, x):
        x = self.features(x)
        x = self.extra_conv(x)
        return x


class RegionProposalNetwork(nn.Module):
    """
    Region Proposal Network - генерує candidate regions для детекції.
    Базується на anchor boxes з різними scales та aspect ratios.
    """
    def __init__(
        self, 
        in_channels: int = 256,
        num_anchors: int = 9,
        feature_stride: int = 16
    ):
        super().__init__()
        self.feature_stride = feature_stride
        self.num_anchors = num_anchors
        
        # 3×3 conv для обробки feature map
        self.conv = nn.Conv2d(in_channels, 512, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)
        
        # Класифікація: objectness score (об'єкт чи фон)
        self.cls_logits = nn.Conv2d(512, num_anchors * 2, kernel_size=1)
        
        # Регресія: зміщення bbox (dx, dy, dw, dh)
        self.bbox_pred = nn.Conv2d(512, num_anchors * 4, kernel_size=1)
        
        # Ініціалізація ваг
        for layer in [self.conv, self.cls_logits, self.bbox_pred]:
            nn.init.normal_(layer.weight, std=0.01)
            nn.init.constant_(layer.bias, 0)
    
    def forward(self, features):
        x = self.conv(features)
        x = self.relu(x)
        
        objectness = self.cls_logits(x)
        bbox_deltas = self.bbox_pred(x)
        
        return objectness, bbox_deltas


class DetectionHead(nn.Module):
    """
    Detection Head для класифікації та уточнення bbox.
    Приймає RoI features та передбачає клас об'єкта і координати bbox.
    """
    def __init__(
        self, 
        in_channels: int = 256,
        num_classes: int = 81,  # 80 класів + background
        roi_size: int = 7
    ):
        super().__init__()
        self.num_classes = num_classes
        
        # RoI pooling для уніфікації розміру features
        self.roi_pool = torchvision.ops.RoIPool(
            output_size=(roi_size, roi_size),
            spatial_scale=1.0/16
        )
        
        # FC layers (як у AlexNet classifier)
        fc_input_size = in_channels * roi_size * roi_size
        self.fc6 = nn.Linear(fc_input_size, 4096)
        self.fc7 = nn.Linear(4096, 4096)
        
        # Класифікація
        self.cls_score = nn.Linear(4096, num_classes)
        
        # Регресія bbox
        self.bbox_pred = nn.Linear(4096, num_classes * 4)
        
        self.dropout = nn.Dropout(0.5)
        
    def forward(self, features, proposals):
        # RoI Pooling
        pooled = self.roi_pool(features, proposals)
        pooled = pooled.flatten(start_dim=1)
        
        # FC layers з dropout
        x = F.relu(self.fc6(pooled))
        x = self.dropout(x)
        x = F.relu(self.fc7(x))
        x = self.dropout(x)
        
        # Передбачення
        cls_scores = self.cls_score(x)
        bbox_deltas = self.bbox_pred(x)
        
        return cls_scores, bbox_deltas


class AlexNetDetector(nn.Module):
    """
    Повна модель AlexNet для Object Detection.
    
    Архітектура:
    1. AlexNet backbone (feature extraction)
    2. RPN (region proposals)
    3. Detection head (classification + bbox regression)
    """
    def __init__(
        self,
        num_classes: int = 81,  # 80 COCO + background
        pretrained_backbone: bool = True,
        anchor_scales: Tuple[float, ...] = (32, 64, 128, 256, 512),
        anchor_ratios: Tuple[float, ...] = (0.5, 1.0, 2.0),
    ):
        super().__init__()
        
        self.num_classes = num_classes
        self.anchor_scales = anchor_scales
        self.anchor_ratios = anchor_ratios
        self.num_anchors = len(anchor_scales) * len(anchor_ratios)
        
        # Компоненти моделі
        self.backbone = AlexNetBackbone(pretrained=pretrained_backbone)
        self.rpn = RegionProposalNetwork(
            in_channels=256,
            num_anchors=self.num_anchors
        )
        self.detection_head = DetectionHead(
            in_channels=256,
            num_classes=num_classes
        )
        
        print(f"✓ AlexNetDetector initialized:")
        print(f"  - Backbone: AlexNet (pretrained={pretrained_backbone})")
        print(f"  - Num classes: {num_classes}")
        print(f"  - Anchors: {self.num_anchors} = {len(anchor_scales)} scales × {len(anchor_ratios)} ratios")
    
    def forward(self, images, targets=None):
        """
        Forward pass моделі.
        
        Training mode: повертає dict з losses
        Inference mode: повертає List[Dict] з predictions
        """
        # Перетворюємо список зображень на batch tensor
        if isinstance(images, list):
            images = torch.stack(images)
        
        # Feature extraction
        features = self.backbone(images)
        
        # Region proposals
        objectness, bbox_deltas = self.rpn(features)
        
        if self.training:
            assert targets is not None, "Targets required for training"
            losses = self._compute_losses(objectness, bbox_deltas, features, targets)
            return losses
        else:
            predictions = self._generate_predictions(objectness, bbox_deltas, features)
            return predictions
    
    def _compute_losses(self, objectness, bbox_deltas, features, targets):
        """Обчислення втрат (буде реалізовано в training loop)"""
        losses = {
            'loss_objectness': torch.tensor(0.0, device=objectness.device),
            'loss_rpn_box_reg': torch.tensor(0.0, device=bbox_deltas.device),
            'loss_classifier': torch.tensor(0.0, device=features.device),
            'loss_box_reg': torch.tensor(0.0, device=features.device),
        }
        return losses
    
    def _generate_predictions(self, objectness, bbox_deltas, features):
        """Генерація predictions для inference"""
        batch_size = features.shape[0]
        predictions = []
        for i in range(batch_size):
            predictions.append({
                'boxes': torch.empty((0, 4), device=features.device),
                'labels': torch.empty((0,), dtype=torch.int64, device=features.device),
                'scores': torch.empty((0,), device=features.device),
            })
        return predictions


def create_alexnet_detector(num_classes: int = 81, pretrained: bool = True) -> AlexNetDetector:
    """
    Factory function для створення AlexNet detector.
    
    Args:
        num_classes: Кількість класів (80 COCO + 1 background = 81)
        pretrained: Використовувати pretrained AlexNet backbone
    """
    model = AlexNetDetector(
        num_classes=num_classes,
        pretrained_backbone=pretrained,
        # Параметри базовані на аналізі з lab1
        anchor_scales=(32, 64, 128, 256, 512),
        anchor_ratios=(0.5, 1.0, 2.0),
    )
    return model


# Тестування архітектури
print("\n" + "="*80)
print("ТЕСТУВАННЯ ALEXNET DETECTOR")
print("="*80 + "\n")

model_alexnet = create_alexnet_detector(num_classes=81, pretrained=True)
model_alexnet = model_alexnet.to(cfg.device)
model_alexnet.eval()

# Тестові дані
test_batch = 2
test_images = torch.randn(test_batch, 3, 800, 1200).to(cfg.device)

print(f"Input shape: {test_images.shape}")

with torch.no_grad():
    features = model_alexnet.backbone(test_images)
    print(f"Backbone output: {features.shape}")
    
    objectness, bbox_deltas = model_alexnet.rpn(features)
    print(f"RPN objectness: {objectness.shape}")
    print(f"RPN bbox_deltas: {bbox_deltas.shape}")

# Статистика параметрів
total_params = sum(p.numel() for p in model_alexnet.parameters())
trainable_params = sum(p.numel() for p in model_alexnet.parameters() if p.requires_grad)

print(f"\nСтатистика моделі:")
print(f"  - Всього параметрів: {total_params:,}")
print(f"  - Trainable параметрів: {trainable_params:,}")
print(f"  - Розмір моделі: ~{total_params * 4 / 1024 / 1024:.2f} MB (float32)")

print("\n" + "="*80)
print("✓ AlexNet Detector успішно створено!")
print("="*80 + "\n")


ТЕСТУВАННЯ ALEXNET DETECTOR

✓ AlexNetDetector initialized:
  - Backbone: AlexNet (pretrained=True)
  - Num classes: 81
  - Anchors: 15 = 5 scales × 3 ratios
Input shape: torch.Size([2, 3, 800, 1200])
Backbone output: torch.Size([2, 256, 24, 36])
RPN objectness: torch.Size([2, 30, 24, 36])
RPN bbox_deltas: torch.Size([2, 60, 24, 36])

Статистика моделі:
  - Всього параметрів: 74,701,103
  - Trainable параметрів: 74,701,103
  - Розмір моделі: ~284.96 MB (float32)

✓ AlexNet Detector успішно створено!

